In [ ]:
import os
import numpy as np
import pandas as pd
# Attempt to import pandas_ta, provide instructions if missing
try:
    import pandas_ta as ta
except ImportError:
    print("="*50)
    print("Please install the 'pandas-ta' library for technical indicators:")
    print("  pip install pandas-ta")
    print("="*50)
    exit() # Exit if pandas_ta is needed but not found

import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
# Import metrics for evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
# Import LSTM, GRU, Dense, Dropout layers
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
# Import EarlyStopping callback and Adam optimizer
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import warnings

# Suppress TensorFlow warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='keras')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Suppress TensorFlow INFO messages

# ------------ Step 1: Choose Company ------------
# Ensure the 'datas/company_data' directory exists and contains CSV files
data_folder = 'datas/company_data'

# --- Create dummy data if folder/files don't exist (Optional: Keep for robustness) ---
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
    print(f"Created data directory: {data_folder}")

dummy_file_path = os.path.join(data_folder, 'DUMMY_historical_data.csv')
if not os.path.exists(dummy_file_path):
    # Check if *any* csv exists before creating dummy data
    existing_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]
    if not existing_files:
        print("Creating dummy data file as no CSVs found...")
        # Create more realistic dummy data with multiple features
        dates_dummy = pd.date_range(end=pd.Timestamp.today() - pd.Timedelta(days=1), periods=500, freq='B')
        price_changes = np.random.randn(500) * 5 + np.sin(np.linspace(0, 20, 500)) * 20
        prices_dummy = 100 + np.cumsum(price_changes)
        prices_dummy = np.maximum(prices_dummy, 10)
        # Simulate other features relative to closing price
        open_dummy = prices_dummy - np.random.rand(500) * 2
        high_dummy = np.maximum(prices_dummy, open_dummy) + np.random.rand(500) * 3
        low_dummy = np.minimum(prices_dummy, open_dummy) - np.random.rand(500) * 3
        volume_dummy = np.random.randint(10000, 1000000, 500)
        dummy_df = pd.DataFrame({
            'fullName': 'DUMMY Corp',
            'symbol': 'DUMMY',
            'numOfTransactions': volume_dummy, # Using numOfTransactions as volume
            'open': open_dummy,
            'high': high_dummy,
            'low': low_dummy,
            'closingPrice': prices_dummy,
            'date': dates_dummy.strftime('%Y-%m-%d')
        })
        dummy_df.to_csv(dummy_file_path, index=False)
        print(f"Dummy file created at: {dummy_file_path}")
# --- End Dummy Data Creation ---

# --- List available companies ---
try:
    available_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]
    if not available_files:
        raise FileNotFoundError(f"No CSV files found in '{data_folder}'. Please add company data CSV files (e.g., 'COMPANY_historical_data.csv').")
    available_companies = sorted([f.replace('_historical_data.csv', '') for f in available_files])
except FileNotFoundError as e:
    print(e)
    exit()
except Exception as e:
    print(f"Error accessing data folder '{data_folder}': {e}")
    exit()


print("Available Companies:")
for idx, company in enumerate(available_companies, start=1):
    print(f"{idx}. {company}")

# --- Interactive company selection ---
while True: # Loop until valid input is received
    choice_str = input(f"Enter the number of the company you want to predict (1-{len(available_companies)}): ")
    try:
        choice = int(choice_str)
        if 1 <= choice <= len(available_companies):
            break # Exit loop if choice is valid
        else:
            print(f"Invalid choice. Please enter a number between 1 and {len(available_companies)}.")
    except ValueError:
        print("Invalid input. Please enter a number.")
# --- End interactive selection ---

selected_company = available_companies[choice - 1]
selected_file = f"{selected_company}_historical_data.csv"
file_path = os.path.join(data_folder, selected_file)

# Define the base features we want to use from the CSV
base_feature_columns = ['open', 'high', 'low', 'closingPrice', 'numOfTransactions']
target_column = 'closingPrice' # The primary column we want to predict/evaluate

try:
    data = pd.read_csv(file_path)
    print(f"\nLoaded data for {selected_company}\n")
    # Check if all required columns exist
    required_csv_columns = ['date'] + base_feature_columns
    missing_cols = [col for col in required_csv_columns if col not in data.columns]
    if missing_cols:
        raise ValueError(f"CSV file '{selected_file}' is missing required columns: {missing_cols}. Found: {list(data.columns)}")
    print("Original data head:")
    print(data.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    exit()
except ValueError as e:
    print(f"Data Error: {e}")
    exit()
except Exception as e:
    print(f"Error loading data from '{file_path}': {e}")
    exit()

# ------------ Step 2: Prepare Data & Add TA Features ------------
print(f"\nPreparing data using features: {base_feature_columns} + TA indicators")
# Ensure data is sorted by date
try:
    data['date'] = pd.to_datetime(data['date'])
except Exception as e:
    print(f"Error converting 'date' column to datetime objects: {e}")
    print("Please ensure the 'date' column is in a recognizable format (e.g., YYYY-MM-DD).")
    exit()

data = data.sort_values('date').reset_index(drop=True) # Reset index after sorting

# Select and ensure numeric types for base feature columns
try:
    data_processed = data[['date'] + base_feature_columns].copy() # Keep date for now
    # Rename columns to lowercase expected by pandas_ta if needed (e.g., 'High' -> 'high')
    data_processed.rename(columns={
        'open': 'open',
        'high': 'high',
        'low': 'low',
        'closingPrice': 'close', # pandas_ta often uses 'close'
        'numOfTransactions': 'volume' # pandas_ta often uses 'volume'
    }, inplace=True)
    # Update base_feature_columns to reflect potential renaming
    base_feature_columns_renamed = [col if col != 'closingPrice' else 'close' for col in base_feature_columns]
    base_feature_columns_renamed = [col if col != 'numOfTransactions' else 'volume' for col in base_feature_columns_renamed]


    for col in base_feature_columns_renamed:
        data_processed[col] = pd.to_numeric(data_processed[col], errors='coerce')

    # Check for NaNs introduced by coercion before adding TA features
    if data_processed[base_feature_columns_renamed].isnull().values.any():
        nan_counts = data_processed[base_feature_columns_renamed].isnull().sum()
        print("Warning: Non-numeric values found in base features and converted to NaN:")
        print(nan_counts[nan_counts > 0])
        # Drop rows with any NaN in the base features before calculating TA
        initial_rows = len(data_processed)
        data_processed.dropna(subset=base_feature_columns_renamed, inplace=True)
        print(f"Dropped {initial_rows - len(data_processed)} rows with NaN values in base features.")
        if data_processed.empty:
             print("Error: No valid numeric data remaining after handling NaNs in base features.")
             exit()

except Exception as e:
    print(f"Error processing base feature columns: {e}")
    exit()

# --- Calculate TA Features using pandas_ta ---
print("Calculating TA features (SMA, RSI, Supertrend)...") # Updated print statement
# Initialize all_feature_columns with renamed base columns first
all_feature_columns = base_feature_columns_renamed.copy()
ta_features_to_add = [] # Initialize empty list for TA feature names to add

try:
    # Calculate SMA (Simple Moving Average) - 10 and 20 periods
    data_processed.ta.sma(length=10, close='close', append=True) # Specify close column
    data_processed.ta.sma(length=20, close='close', append=True) # Specify close column

    # Calculate RSI (Relative Strength Index) - 14 periods
    data_processed.ta.rsi(length=14, close='close', append=True) # Specify close column

    # Calculate Supertrend (using default period=7, multiplier=3)
    # Requires high, low, close columns
    st_result = data_processed.ta.supertrend(append=True) # Appends SUPERT_7_3.0, SUPERTd_7_3.0, etc.

    # Define the TA features we want to keep
    # Keep SMA, RSI, and the Supertrend line + direction
    ta_features_to_add = ['SMA_10', 'SMA_20', 'RSI_14', 'SUPERT_7_3.0', 'SUPERTd_7_3.0'] # Changed from MACD to Supertrend

    # Check if the expected columns were added by supertrend
    missing_st_cols = [col for col in ta_features_to_add if col.startswith('SUPERT') and col not in data_processed.columns]
    if missing_st_cols:
        print(f"Warning: Expected Supertrend columns not found: {missing_st_cols}. Available columns: {data_processed.columns}")
        # Adjust ta_features_to_add if columns are missing
        ta_features_to_add = [col for col in ta_features_to_add if not col.startswith('SUPERT')]


    # Update the list of all features to include TA features
    all_feature_columns = base_feature_columns_renamed + ta_features_to_add
    print(f"Successfully added TA features. Final features: {all_feature_columns}")

    # --- Handle NaNs introduced by TA calculations ---
    # TA indicators create NaNs at the beginning
    if data_processed[all_feature_columns].isnull().values.any():
        print("Handling NaNs introduced by TA indicators...")
        initial_rows_ta = len(data_processed)
        # Drop rows with any NaN in *any* of the final feature columns
        data_processed.dropna(subset=all_feature_columns, inplace=True)
        print(f"Dropped {initial_rows_ta - len(data_processed)} rows due to TA indicator NaNs.")
        if data_processed.empty:
             print("Error: No data remaining after handling TA indicator NaNs.")
             exit()
        # Reset index after dropping rows
        data_processed = data_processed.reset_index(drop=True)

except Exception as e:
    print(f"Error calculating or processing TA features: {e}")
    # Potentially try installing pandas_ta if it's a related error
    if "AttributeError" in str(e) and ".ta" in str(e):
         print("\nHint: Ensure 'pandas-ta' is installed (`pip install pandas-ta`) and imported correctly.")
    # If TA fails, continue with base features only
    print("Proceeding with base features only due to TA calculation error.")
    all_feature_columns = base_feature_columns_renamed # Revert to base columns if TA failed

# Extract the final processed feature data and corresponding dates
# Use the potentially updated all_feature_columns list
features_final = data_processed[all_feature_columns].values
dates_all = data_processed['date'] # Dates corresponding to the final features
n_features = features_final.shape[1] # Number of final features
print(f"Using {n_features} features.")
print("Data head after processing:")
print(data_processed[all_feature_columns].head())


# Find the index of the target column ('close' after renaming) within the *final* feature set
try:
    target_col_index = all_feature_columns.index('close') # Use 'close' after renaming
    print(f"Target column 'close' is at index {target_col_index} in the final feature set.")
except ValueError:
    print(f"Critical Error: Target column 'close' not found in final feature list: {all_feature_columns}")
    exit()


# Define parameters here for clarity
time_step = 30 # Number of past days to use for predicting the next day
prediction_days = 60 # Days to predict into the future (now 60)
overlap_days = 30 # Days to overlap for plotting

# Check if enough data is available AFTER potential row drops from TA NaNs
required_data_points = time_step + 1 # Need at least time_step + 1 for one training sample
if len(features_final) < required_data_points:
     print(f"Error: Not enough data for {selected_company} after cleaning and TA. Need more than {time_step} data points, found {len(features_final)}.")
     exit()
# Also check if enough data for the overlap sequence generation
if len(features_final) < time_step + overlap_days:
     print(f"Warning: Not enough data for full {overlap_days}-day overlap ({len(features_final)} points < {time_step + overlap_days} required). Overlap plot might be shorter.")


# Scale all final features together
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_features = scaler.fit_transform(features_final)

# --- Split data into training and test sets (e.g., 80% train, 20% test) ---
training_size = int(len(scaled_features) * 0.80)
test_size = len(scaled_features) - training_size
train_data, test_data = scaled_features[0:training_size,:], scaled_features[training_size:len(scaled_features),:]
print(f"\nData Split: Training size = {training_size}, Test size = {test_size}")

# Function to create dataset for multi-feature input
# X = sequences of shape (time_step, n_features)
# y = next day's features of shape (n_features)
def create_multi_feature_dataset(dataset, time_step=30): # Use updated time_step
    X, y = [], []
    if len(dataset) <= time_step:
        return np.array([]), np.array([])
    for i in range(len(dataset) - time_step):
        # Input sequence (past 'time_step' days of all features)
        X.append(dataset[i:(i + time_step), :])
        # Output (all features for the next day)
        y.append(dataset[i + time_step, :])
    if not X or not y:
        return np.array([]), np.array([])
    return np.array(X), np.array(y)

# Create training and test datasets using the new time_step
X_train, y_train = create_multi_feature_dataset(train_data, time_step)
X_test, y_test = create_multi_feature_dataset(test_data, time_step)

# Check if datasets were created successfully
if X_train.size == 0 or y_train.size == 0:
    print(f"Error: Could not create training dataset. Check training_size ({training_size}) and time_step ({time_step}).")
    exit()
if X_test.size == 0 or y_test.size == 0:
    print(f"Error: Could not create test dataset. Check test_size ({test_size}) and time_step ({time_step}).")
    exit()

# Reshape is already done by create_multi_feature_dataset, X shape is (samples, time_steps, n_features)

print(f"\nData Preparation Summary:")
print(f"Total historical data points used (after TA NaNs): {len(features_final)}")
print(f"Time step for RNN: {time_step}")
print(f"Number of features: {n_features}")
print(f"Shape of training data X_train: {X_train.shape}") # (samples, time_step, n_features)
print(f"Shape of training data y_train: {y_train.shape}") # (samples, n_features)
print(f"Shape of test data X_test: {X_test.shape}")       # (samples, time_step, n_features)
print(f"Shape of test data y_test: {y_test.shape}")         # (samples, n_features)

# ------------ Step 3: Build and Train Unidirectional Multi-Feature Model with TA ------------
print("\nBuilding and training Unidirectional Multi-Feature LSTM + GRU model (with TA features)...")
model = Sequential([
    # Input shape now includes n_features (base + TA) and updated time_step
    LSTM(units=80, return_sequences=True, input_shape=(time_step, n_features)), # Use updated time_step
    Dropout(0.25),
    GRU(units=60),
    Dropout(0.25),
    Dense(units=30),
    Dense(units=15),
    # Output layer predicts all features for the next time step
    Dense(units=n_features)
])

# Print model summary to verify architecture
model.summary()

# Define optimizer with a specific learning rate
learning_rate = 0.001
optimizer = Adam(learning_rate=learning_rate)

model.compile(optimizer=optimizer, loss='mean_squared_error') # Use MSE for training loss

# Define Early Stopping callback
# Monitors validation loss, stops if no improvement after 'patience' epochs, restores best weights
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model using training data and validate on test data
epochs = 100 # Increase epochs, let EarlyStopping find the best one
batch_size = 32
history = model.fit(X_train, y_train, # y_train now has n_features
                    validation_data=(X_test, y_test), # y_test now has n_features
                    epochs=epochs,
                    batch_size=batch_size,
                    callbacks=[early_stopping], # Add early stopping
                    verbose=1)
print("Model training finished.")
print(f"Training stopped after {len(history.history['loss'])} epochs due to EarlyStopping.")


# ------------ Step 3.5: Evaluate Model Accuracy on Test Set (Focus on Closing Price) ------------
print("\nEvaluating model accuracy on test data (focusing on closing price)...")

# Make predictions on the test set (predicts all features)
test_predictions_scaled = model.predict(X_test) # Shape: (n_samples, n_features)

# Inverse transform predictions and actual values to original scale
# Scaler expects input shape (n_samples, n_features)
test_predictions_actual = scaler.inverse_transform(test_predictions_scaled)
y_test_actual_all_features = scaler.inverse_transform(y_test) # y_test shape is (n_samples, n_features)

# Extract the 'closingPrice' column for evaluation using target_col_index
predicted_closing_price_test = test_predictions_actual[:, target_col_index]
actual_closing_price_test = y_test_actual_all_features[:, target_col_index]

# Calculate evaluation metrics for closing price on TEST SET
if predicted_closing_price_test.shape != actual_closing_price_test.shape:
     print(f"Warning: Shape mismatch between test predictions {predicted_closing_price_test.shape} and actual values {actual_closing_price_test.shape}. Cannot calculate metrics accurately.")
else:
    try:
        # RMSE
        rmse = np.sqrt(mean_squared_error(actual_closing_price_test, predicted_closing_price_test))
        print(f"  Closing Price RMSE (Test Set): {rmse:.4f}")

        # MAE
        mae = mean_absolute_error(actual_closing_price_test, predicted_closing_price_test)
        print(f"  Closing Price MAE (Test Set):  {mae:.4f}")

        # MAPE - Handle potential zero values
        epsilon = 1e-8
        safe_actual_closing_price_test = np.maximum(actual_closing_price_test, epsilon)
        mape = mean_absolute_percentage_error(safe_actual_closing_price_test, predicted_closing_price_test) * 100
        print(f"  Closing Price MAPE (Test Set): {mape:.2f}%")

        # Accuracy Percentage (100 - MAPE)
        accuracy_percent = 100.0 - mape
        print(f"  Closing Price Accuracy (Test Set, %): {accuracy_percent:.2f}%")

    except Exception as e:
        print(f"Error calculating closing price metrics: {e}")


# ------------ Step 4: Continuous Prediction for Overlap + Future Plot ------------
# Predict for `overlap_days` + `prediction_days` (now 30 + 60 = 90 days)
total_prediction_span = overlap_days + prediction_days

# Determine the starting sequence for this combined prediction
# Need the sequence ending `overlap_days` before the last historical point.
start_index_for_input = len(scaled_features) - overlap_days - time_step # Use updated time_step
end_index_for_input = len(scaled_features) - overlap_days

if start_index_for_input < 0:
    print(f"Error: Not enough historical data to create the initial overlap sequence.")
    print(f"Need {overlap_days + time_step} days, but only have {len(scaled_features)} after cleaning.")
    # Fallback: Start prediction sequence from the beginning if insufficient history for overlap
    start_index_for_input = 0
    end_index_for_input = time_step # Use updated time_step
    print(f"Warning: Starting overlap prediction sequence from the beginning of the data.")

initial_input_seq_scaled = scaled_features[start_index_for_input:end_index_for_input]

print(f"\nGenerating {total_prediction_span}-day prediction ({overlap_days}-day overlap + {prediction_days}-day future)...")

overlap_future_predictions_scaled = [] # Use a list to append
# Reshape initial sequence for model input [batch_size, time_steps, features]
current_input_seq = initial_input_seq_scaled.reshape(1, time_step, n_features) # Use updated time_step

# Predict future `total_prediction_span` days
for i in range(total_prediction_span):
    # Predict the next set of features (scaled)
    next_features_scaled = model.predict(current_input_seq, verbose=0)[0] # Get the array of features
    overlap_future_predictions_scaled.append(next_features_scaled)

    # Prepare the next input sequence:
    next_seq_base = current_input_seq[:, 1:, :] # Shape (1, time_step-1, n_features)
    next_features_reshaped = next_features_scaled.reshape(1, 1, n_features)
    current_input_seq = np.append(next_seq_base, next_features_reshaped, axis=1) # Shape (1, time_step, n_features)


# Inverse transform the scaled overlap+future predictions (all features)
# Input shape for inverse_transform needs to be (n_samples, n_features)
overlap_future_predictions_actual = scaler.inverse_transform(np.array(overlap_future_predictions_scaled))
print("Overlap+Future prediction generation finished.")

# Extract the predicted closing prices for this period
overlap_future_predicted_closing_prices = overlap_future_predictions_actual[:, target_col_index]


# ------------ Step 5: Generate Dates for Overlap + Future Plotting ------------
# The prediction starts `overlap_days` before the end of historical data.
# The date for the first prediction corresponds to the day *after* the last day in the initial input sequence.
# The last day in the initial input sequence is at index `end_index_for_input - 1`.
# So the first prediction corresponds to the date at index `end_index_for_input`.
try:
    # Ensure end_index_for_input is within bounds of the 'dates_all' Series (post-TA NaN removal)
    if end_index_for_input >= len(dates_all):
         raise IndexError(f"Calculated end index for input ({end_index_for_input}) is out of bounds for historical dates (length {len(dates_all)}).")
    prediction_start_date = dates_all.iloc[end_index_for_input]
except IndexError as e:
    print(f"Error determining overlap prediction start date: {e}")
    print("Using last historical date + 1 day as fallback (plot might be misaligned).")
    # Ensure dates_all[-1] exists
    if len(dates_all) > 0:
        prediction_start_date = dates_all.iloc[-1] + pd.Timedelta(days=1)
    else:
        print("Critical Error: No dates available to determine prediction start date.")
        exit()


# Determine frequency
is_business_day = False # Default to calendar days unless proven otherwise
if len(dates_all) > 50: # Check only if enough data exists
   is_business_day = all(d.weekday() < 5 for d in dates_all.iloc[-min(50, len(dates_all)):])
freq = 'B' if is_business_day else 'D'
print(f"Using frequency '{freq}' for future prediction dates based on historical data.")

# Generate dates for the overlap + future prediction span (now 90 days total)
overlap_future_dates = pd.date_range(start=prediction_start_date, periods=total_prediction_span, freq=freq)

# Adjust future dates if frequency causes issues
if len(overlap_future_dates) != total_prediction_span:
     print(f"Warning: Overlap+Future date range generation with freq='{freq}' resulted in {len(overlap_future_dates)} dates instead of {total_prediction_span}. Trying opposite frequency.")
     alt_freq = 'D' if freq == 'B' else 'B'
     overlap_future_dates = pd.date_range(start=prediction_start_date, periods=total_prediction_span, freq=alt_freq)
     if len(overlap_future_dates) != total_prediction_span:
          print(f"Warning: Overlap+Future date range generation still resulted in {len(overlap_future_dates)} dates with freq='{alt_freq}'. Using calendar days ('D') as final fallback.")
          overlap_future_dates = pd.date_range(start=prediction_start_date, periods=total_prediction_span, freq='D')


print(f"\nOverlap+Future Prediction Date Range (for plotting):")
if not overlap_future_dates.empty:
    print(f"Starts: {overlap_future_dates[0].strftime('%Y-%m-%d')}") # Use index 0
    print(f"Ends:   {overlap_future_dates[-1].strftime('%Y-%m-%d')}") # Use index -1
    print(f"Number of prediction dates generated: {len(overlap_future_dates)}")
else:
    print("Error: Could not generate overlap+future prediction dates.")


# ------------ Step 6: Plotting ------------
print("\nPlotting results (showing Closing Price with Overlap)...")
plt.figure(figsize=(15, 8))

# Extract actual historical closing prices for plotting (from data post-TA NaN removal)
actual_historical_closing_prices = features_final[:, target_col_index]

# Plot actual historical closing price (full history post-TA NaN removal)
plt.plot(dates_all, actual_historical_closing_prices, color='royalblue', label='Actual Historical Closing Price', linewidth=2, alpha=0.8)

# Plot the combined predicted closing price (overlap + future)
if len(overlap_future_dates) == len(overlap_future_predicted_closing_prices):
    # Update label to reflect 60-day future prediction
    plt.plot(overlap_future_dates, overlap_future_predicted_closing_prices, color='tomato', marker='.', markersize=5, linestyle='--',
             label=f'Predicted Closing Price ({overlap_days}-day Overlap + {prediction_days}-day Future)', linewidth=1.5)
else:
    print(f"Critical Warning: Mismatch between number of overlap+future dates ({len(overlap_future_dates)}) and predictions ({len(overlap_future_predicted_closing_prices)}). Cannot plot overlap prediction accurately.")


# Common plotting elements
# Update title to reflect TA features used and time_step
plt.title(f'{selected_company} Stock Price Prediction - Unidirectional + TA Features (SMA, RSI, Supertrend, Timestep={time_step})', fontsize=16, weight='bold') # Updated title
plt.xlabel('Date', fontsize=12)
plt.ylabel('Stock Price (Closing)', fontsize=12) # Label y-axis clearly
plt.xticks(rotation=45, ha='right')
plt.legend(fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()
print("Plot displayed.")

# ------------ Step 7: Display Predicted Prices (Optional) ------------
# Display only the purely future part of the prediction (last `prediction_days`, now 60)
if len(overlap_future_dates) >= prediction_days and len(overlap_future_predicted_closing_prices) >= prediction_days: # Check lengths match
    # Get the dates corresponding to the purely future predictions
    future_dates_display = overlap_future_dates[-prediction_days:]
    # Get the corresponding closing price predictions
    future_closing_prices_display = overlap_future_predicted_closing_prices[-prediction_days:]

    future_predictions_df = pd.DataFrame({
        'Date': future_dates_display,
        'Predicted Closing Price': future_closing_prices_display.flatten()
    })

    # Update print statement to reflect 60 days
    print(f"\nPredicted Closing Prices for the Next {prediction_days} Days:")
    future_predictions_df['Date'] = future_predictions_df['Date'].dt.strftime('%Y-%m-%d')
    future_predictions_df['Predicted Closing Price'] = future_predictions_df['Predicted Closing Price'].map('{:.2f}'.format)
    print(future_predictions_df.to_string(index=False))
else:
    print("\nWarning: Could not display future predicted closing prices due to length mismatch or insufficient predictions/dates.")
    print(f"Need {prediction_days} predictions, found {len(overlap_future_predicted_closing_prices)}.")
    print(f"Need {prediction_days} overlap+future dates, found {len(overlap_future_dates)}.")

